# Variational Autoencoder (VAE) on MNIST — Beginner Tutorial

This tutorial uses the VAE implemented in this repository (`models/vae.py`) and runs training/evaluation on GPU.
It is designed for beginners with minimal coding experience.

What you will learn:
- Load and preprocess MNIST data.
- Use the repo's convolutional VAE.
- Train on GPU and monitor the loss.
- Reconstruct digits and generate new samples.


## Requirements

- Python 3
- TensorFlow 2.x (includes Keras)

If using Google Colab, you already have these. On a local machine, install TensorFlow 2.x following the official guide if needed.


In [1]:
# Imports and basic setup
import keras
import sys, os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# Make results a bit more repeatable (optional)
tf.random.set_seed(42)
np.random.seed(42)


## GPU Check

We train and evaluate on GPU. If no GPU is found, the cell raises an error.


In [ ]:
# Ensure a GPU is available
gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError('No GPU detected by TensorFlow. Run the GPU Diagnostics cell above and ensure your kernel has TF with GPU support, compatible drivers, CUDA/cuDNN.')
# Optional: set memory growth
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass
print('Using GPU:', gpus[0].name)


## Load MNIST

MNIST contains 70,000 grayscale images of handwritten digits (28×28). We normalize to [0, 1], add a channel dimension, and resize to 128×128 to match the repo VAE.


In [ ]:
# Download and prepare the data (resize to 128x128)
(x_train, _), (x_test, _) = keras.datasets.mnist.load_data()
x_train = (x_train.astype('float32') / 255.0)[..., None]  # (60000, 28, 28, 1)
x_test  = (x_test.astype('float32')  / 255.0)[..., None]  # (10000, 28, 28, 1)

x_train = tf.image.resize(x_train, (128, 128)).numpy()
x_test  = tf.image.resize(x_test,  (128, 128)).numpy()

print('Train:', x_train.shape, ' Test:', x_test.shape)


## Peek at the data

Let’s visualize a small grid of random training digits.


In [ ]:
# Show a 5x5 grid of random digits
idx = np.random.choice(len(x_train), 25, replace=False)
imgs = x_train[idx]  # (25, 128, 128, 1)

plt.figure(figsize=(5, 5))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.imshow(imgs[i].squeeze(), cmap='gray')
    plt.axis('off')
plt.tight_layout()
plt.show()


# VAE

In [ ]:
import tensorflow as tf

def conv3x3(channels, stride=1, **kwargs):
    return keras.layers.Conv2D(
        channels, (3, 3),
        strides=stride,
        padding='same',
        kernel_initializer='he_normal',
        **kwargs
    )

# BatchNorm + ReLU
class BNReLU(keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.bn = keras.layers.BatchNormalization(name='bn')

    def call(self, inputs, training=None):
        y = self.bn(inputs, training=training)
        return tf.nn.relu(y)

# bloque conv
class ConvBlock(keras.layers.Layer):
    def __init__(self, channels, **kwargs):
        super().__init__(**kwargs)
        self.conv_1 = keras.layers.Conv2D(
            channels, (3, 3), strides=2, padding='same', kernel_initializer='he_normal', **kwargs
        )
        self.conv_2 = keras.layers.Conv2D(
            channels, (3, 3), strides=1, padding='same', kernel_initializer='he_normal', **kwargs
        )

    def call(self, inputs, training=None):
        y = self.conv_1(inputs)
        y = self.conv_2(y)
        return y

# encoder
class Encoder(keras.layers.Layer):
    def __init__(self, channels, target_dimension, **kwargs):
        super().__init__(**kwargs)
        self.conv1 = ConvBlock(channels[0])  # 64
        self.bn_relu_1 = BNReLU()
        self.conv2 = ConvBlock(channels[1])  # 32
        self.bn_relu_2 = BNReLU()
        self.conv3 = ConvBlock(channels[2])  # 16
        self.bn_relu_3 = BNReLU()
        self.conv4 = ConvBlock(channels[3])  # 8x8x64
        self.bn_relu_4 = BNReLU()
        self.flatten = keras.layers.Flatten()
        self.dense_mu = keras.layers.Dense(target_dimension)
        self.dense_log_var = keras.layers.Dense(target_dimension)

    def call(self, inputs, training=None):
        # input: [128,128,1]
        y = self.conv1(inputs, training=training)      # 64x64
        y = self.bn_relu_1(y, training=training)
        y = self.conv2(y, training=training)           # 32x32
        y = self.bn_relu_2(y, training=training)
        y = self.conv3(y, training=training)           # 16x16
        y = self.bn_relu_3(y, training=training)
        y = self.conv4(y, training=training)           # 8x8
        y = self.bn_relu_4(y, training=training)
        y = self.flatten(y)                             # 8*8*64 = 4096
        mu = self.dense_mu(y)
        log_var = self.dense_log_var(y)
        return tf.concat([mu, log_var], axis=1)        # [mu | logvar]

class Decoder(keras.layers.Layer):
    def __init__(self, channels, **kwargs):
        super().__init__(**kwargs)
        self.dense_1 = keras.layers.Dense(4096)
        self.conv1 = keras.layers.Conv2DTranspose(channels[0], 3, strides=2, padding='same')
        self.bn_relu_1 = BNReLU()
        self.conv2 = keras.layers.Conv2DTranspose(channels[1], 3, strides=2, padding='same')
        self.bn_relu_2 = BNReLU()
        self.conv3 = keras.layers.Conv2DTranspose(channels[2], 3, strides=2, padding='same')
        self.bn_relu_3 = BNReLU()
        self.conv4 = keras.layers.Conv2DTranspose(channels[3], 3, strides=2, padding='same')
        self.bn_relu_4 = BNReLU()
        self.conv5 = keras.layers.Conv2D(1, (1, 1))
        self.sigmoid = keras.activations.sigmoid

    def call(self, inputs, training=None):
        y = self.dense_1(inputs)
        y = tf.reshape(y, (-1, 8, 8, 64))
        y = self.conv1(y, training=training) ; y = self.bn_relu_1(y, training=training)  # 16
        y = self.conv2(y, training=training) ; y = self.bn_relu_2(y, training=training)  # 32
        y = self.conv3(y, training=training) ; y = self.bn_relu_3(y, training=training)  # 64
        y = self.conv4(y, training=training) ; y = self.bn_relu_4(y, training=training)  # 128
        y = self.conv5(y)
        return self.sigmoid(y)

class VAE(keras.Model):
    def __init__(self, channels, **kwargs):
        super().__init__(**kwargs)
        self.encoder = Encoder(channels, 128)
        self.decoder = Decoder(channels[::-1])  # lista invertida

        # capa sin pesos para aplanar salida del decoder
        self._flatten = keras.layers.Flatten()

    def sampling(self, mu_log_var):
        mu, log_var = tf.split(mu_log_var, 2, axis=1)
        epsilon = tf.random.normal(tf.shape(mu), mean=0., stddev=1.)
        return mu + tf.exp(0.5 * log_var) * epsilon

    def call(self, inputs, training=None):
        mu_log_var = self.encoder(inputs, training=training)
        z = self.sampling(mu_log_var)
        x = self.decoder(z, training=training)
        x = self._flatten(x)
        return tf.concat([mu_log_var, x], axis=1)


## Build the VAE

Instantiate the convolutional VAE. It expects input images of shape 128×128×1 and uses a 128‑D latent space.


In [ ]:
input_shape = (128, 128, 1)
channels = [32, 64, 64, 64]  # as used in train.py

model = VAE(channels)
# Build the model by calling it once on an Input tensor
_ = model(keras.Input(shape=input_shape), training=False)
model.summary()


## Define Loss Functions

In [ ]:
import tensorflow as tf

def vae_reconstruction_loss(x_true, x_pred):
    #r_loss = tf.reduce_mean(tf.square(x_true-x_pred), axis = [1,2])
    r_loss= tf.reduce_sum(tf.keras.losses.binary_crossentropy(x_true, x_pred), axis = [1,2])
    return r_loss

def vae_kl_loss(mu, log_var):
    kl_loss = -0.5 * tf.reduce_sum(1 + log_var - tf.square(mu) - tf.exp(log_var), axis = 1)
    return kl_loss

def vae_loss(y_true, y_pred):
    mu_log_var = tf.slice(y_pred, [0,0],[-1,256])
    x = tf.slice(y_pred, [0,256],[-1,-1])
    x_pred = tf.reshape(x, (-1, 128,128,1))
    mu, log_var = tf.split(mu_log_var, 2, axis = 1)
    r_loss = tf.reduce_mean(vae_reconstruction_loss(y_true, x_pred))
    kl_loss = tf.reduce_mean(vae_kl_loss(mu, log_var))
    return r_loss + kl_loss

## Compile and data pipeline

We use the repo loss `utils_vae.losses.vae_loss`, which expects targets equal to inputs. So, our dataset yields pairs `(image, image)`.


In [ ]:
# MNIST en uint8, SIN normalizar ni redimensionar aún
(x_train, _), (x_test, _) = keras.datasets.mnist.load_data()

# añade canal para que quede (H,W,1) pero sigue en uint8
x_train = x_train[..., None]
x_test  = x_test[..., None]

batch_size = 128

def make_ds(x, training=True):
    ds = tf.data.Dataset.from_tensor_slices(x)

    # Todas las transformaciones dentro del pipeline (sin duplicar arrays)
    def _prep(img):
        # resize -> float32 -> normaliza; produce (img, img) como target=entrada
        img = tf.image.resize(img, (128, 128))
        img = tf.cast(img, tf.float32) / 255.0
        return img, img

    ds = ds.map(_prep, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(8192, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size, drop_remainder=False).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_ds(x_train, training=True)
test_ds  = make_ds(x_test,  training=False)

# compilar y entrenar
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss=vae_loss)  # tu loss


## Train the VAE (on GPU)

We train for a few epochs. You can increase `epochs` for better reconstructions.


In [ ]:
epochs = 15
with tf.device('/GPU:0'):
    history = model.fit(train_ds, validation_data=test_ds, epochs=epochs, verbose=1)

# Plot total training loss
plt.figure(figsize=(6, 4))
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('VAE training loss')
plt.show()


## Reconstruct test images

We pass test images through the VAE and compare inputs vs. reconstructions.


In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

# --- RECONSTRUCTIONS (SAFE VERSION) ---
n = 10

# Take the first n examples
test_examples = x_test[:n]

# Resize and normalize (match model input format)
test_examples = tf.image.resize(test_examples, (128, 128))
test_examples = tf.cast(test_examples, tf.float32) / 255.0

# Run prediction on GPU with small batch size to avoid OOM
with tf.device('/GPU:0'):
    preds = model.predict(test_examples, batch_size=4, verbose=0)

# Extract reconstructed images from model output [mu | logvar | flattened image]
x_flat = preds[:, 256:]                                # skip mu and logvar
recons = tf.reshape(x_flat, (-1, 128, 128, 1))          # (n, 128, 128, 1)

# --- VISUALIZATION ---
plt.figure(figsize=(2 * n, 4))
for i in range(n):
    # Original input
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(test_examples[i].numpy().squeeze(), cmap='gray')
    plt.title('Input')
    plt.axis('off')

    # Reconstruction
    ax = plt.subplot(2, n, n + i + 1)
    plt.imshow(recons[i].numpy().squeeze(), cmap='gray')
    plt.title('Recon')
    plt.axis('off')

plt.tight_layout()
plt.show()


## Generate new digits from the latent space

We sample random latent vectors `z ~ N(0, I)` and decode them into digits.


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# --- LATENT SPACE SAMPLING AND GENERATION ---
num_samples = 16
latent_dim = 128  # must match your VAE latent size

z_samples = np.random.normal(size=(num_samples, latent_dim)).astype('float32')

# Call the decoder layer directly; returns a Tensor
with tf.device('/GPU:0'):
    generated = model.decoder(z_samples, training=False)  # (N, 128, 128, 1)

# --- VISUALIZATION ---
rows, cols = 4, 4
plt.figure(figsize=(2.5 * cols, 2.5 * rows))
for i in range(num_samples):
    plt.subplot(rows, cols, i + 1)
    plt.imshow(generated[i].numpy().squeeze(), cmap='gray')
    plt.axis('off')
plt.suptitle('Random samples from the latent space', fontsize=14)
plt.tight_layout()
plt.show()


## Next steps (optional)

- Train longer (e.g., 30–50 epochs) for better reconstructions.
- Adjust channel sizes or add dropout/batch norm for experimentation.
- Try Fashion‑MNIST with the same preprocessing (resize to 128×128).
